# Project Bài 2 · Bộ lọc tin rác bằng kNN tự cài đặt

**Học máy cơ bản · Bài 2 · PGS.TS Nguyễn Văn Hậu**



In [23]:
# Thiết lập: chạy ô này trước
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
TEP = ROOT / 'data' / 'SMSSpamCollection'
assert TEP.exists(), 'Mở notebook từ thư mục bài (có thư mục data/ chứa tệp SMSSpamCollection).'

def kiem(ten, dieu_kien, goi_y=''):
    # Ô tự kiểm: in ✅ khi đúng; báo lỗi kèm gợi ý khi sai.
    if not dieu_kien:
        raise AssertionError(f'❌ {ten}. Gợi ý: {goi_y}')
    print(f'✅ {ten}')

## Bước 1 · Đọc dữ liệu, bỏ tin trùng (Chặng 1)


In [24]:
goc = pd.read_csv(TEP, sep='\t', names=['label', 'text'], quoting=3)
goc['source_record'] = np.arange(1, len(goc) + 1)
# TODO: tạo cột chuan_so_trung theo ba bước ở trên, rồi bỏ trùng (giữ bản đầu tiên) thành bảng d
goc['chuan_so_trung'] = (
    goc['text']
    .str.lower()
    .str.strip()
    .str.replace(r'\s+', ' ', regex=True)
)
d = goc.drop_duplicates(subset='chuan_so_trung', keep='first').copy()

d['y'] = (d.label == 'spam').astype(int)
print('Tệp gốc:', len(goc), 'dòng · sau bỏ trùng:', len(d), 'tin ·', d.y.sum(), 'spam')

Tệp gốc: 5574 dòng · sau bỏ trùng: 5159 tin · 642 spam


In [7]:
# TỰ KIỂM · bước 1
kiem('Tệp gốc có 5.574 dòng', len(goc) == 5574, 'Đọc với sep="\\t", quoting=3, không có dòng tiêu đề.')
kiem('Còn 5.159 tin sau bỏ trùng', len(d) == 5159, 'Gộp khoảng trắng bằng regex \\s+, strip, casefold; drop_duplicates giữ bản đầu.')
kiem('642 tin spam', int(d.y.sum()) == 642, 'Nhãn spam là chuỗi "spam".')
kiem('Không nhóm trùng nào mang hai nhãn', goc.groupby('chuan_so_trung').label.nunique().max() == 1, '')

✅ Tệp gốc có 5.574 dòng


AssertionError: ❌ Còn 5.159 tin sau bỏ trùng. Gợi ý: Gộp khoảng trắng bằng regex \s+, strip, casefold; drop_duplicates giữ bản đầu.

## Bước 2 · Hai đặc trưng và ba tập (Chặng 1)

- `n_chars` = số ký tự của văn bản **gốc** (gồm dấu cách, dấu câu): `len(text)`.
- `n_digits` = số ký tự là chữ số: `c.isdigit()`.
- `digit_fraction` = `n_digits / n_chars`.

Chia ba tập **sau** khi bỏ trùng, có phân tầng theo nhãn, seed 42 (đoạn mã chia đã cho sẵn để mọi người có cùng ba tập).

In [8]:
# TODO: ba cột n_chars, n_digits, digit_fraction như mô tả
d['n_chars'] = d['text'].str.len()
# Số chữ số
d['n_digits'] = d['text'].str.count(r'\d')
# Tỷ lệ chữ số
d['digit_fraction'] = d['n_digits']/d['n_chars']

from sklearn.model_selection import train_test_split
# Chia lần 1
tr, con_lai = train_test_split(d.index, test_size=.4, stratify=d.y, random_state=42)
# Chia lần 2
va, te = train_test_split(con_lai, test_size=.5, stratify=d.loc[con_lai, 'y'], random_state=42)
FEATURES = ['n_chars', 'digit_fraction']
X_tr, y_tr, ma_tr = d.loc[tr, FEATURES].to_numpy(), d.loc[tr, 'y'].to_numpy(), d.loc[tr, 'source_record'].to_numpy()
X_va, y_va = d.loc[va, FEATURES].to_numpy(), d.loc[va, 'y'].to_numpy()
X_te, y_te = d.loc[te, FEATURES].to_numpy(), d.loc[te, 'y'].to_numpy()
print('train / validation / test:', len(tr), len(va), len(te), '· spam:', y_tr.sum(), y_va.sum(), y_te.sum())

train / validation / test: 3093 1031 1032 · spam: 383 128 128


In [9]:
# TỰ KIỂM · bước 2 (mốc M1)
r3 = d.loc[d.source_record == 3].iloc[0]
kiem('Bản ghi 3 có toạ độ (155; 25/155)', r3.n_chars == 155 and r3.n_digits == 25 and np.isclose(r3.digit_fraction, 25 / 155),
     'Đếm cả dấu cách và dấu câu; mỗi chữ số đếm một lần (2005 là 4 chữ số).')
kiem('Ba tập 3.095 / 1.032 / 1.032', (len(tr), len(va), len(te)) == (3095, 1032, 1032), 'Không sửa đoạn mã chia; chia sau khi bỏ trùng.')
kiem('Số spam 385 / 129 / 128', (y_tr.sum(), y_va.sum(), y_te.sum()) == (385, 129, 128), 'Kiểm lại cột y.')

✅ Bản ghi 3 có toạ độ (155; 25/155)


AssertionError: ❌ Ba tập 3.095 / 1.032 / 1.032. Gợi ý: Không sửa đoạn mã chia; chia sau khi bỏ trùng.

**Trả lời câu hỏi M1**
Cần bỏ các tin nhắn trùng lặp trước khi chia để tránh cùng một nội dung xuất hiện ở cả Train và Validation/Test , khiến kết quả đánh giá không khách quan . digit_fraction chỉ biểu thị tỷ lệ chữ số trong tổng số ký tự của tin nhắn , chứ không cho biết xác suất tin đó là spam . Đây chỉ là một đặc trưng đầu vào để mô tả hình kNN sử dụng khi phân loại ham/spam.

**Câu hỏi M1 (trả lời 2–3 câu trong ô Markdown):** vì sao phải bỏ trùng *trước* khi chia tập? Vì sao `digit_fraction` không phải xác suất một tin là spam?

## Bước 3 · Tự viết kNN (Chặng 2)

Viết `knn_du_doan(X_train, y_train, ma_train, X_moi, k, sigma=None)` trả về ba mảng: dự đoán (0/1), tỷ lệ phiếu spam, và **mã bản ghi** của k láng giềng cho từng tin mới.

- Với mỗi tin mới `q`: nếu `sigma` là None, `d² = (x₁ − q₁)² + (x₂ − q₂)²`; nếu có `sigma`, chia từng hiệu cho `sigma[j]` trước khi bình phương (đó chính là Euclid sau chuẩn hoá: trừ cùng μ triệt tiêu trong hiệu).
- Xếp theo d²; **đồng d² thì mã bản ghi nhỏ hơn đứng trước**: `np.lexsort((ma_train, d2))`.
- Spam nếu số phiếu spam > k/2. **Không dùng scikit-learn ở bước này.**

In [10]:
def knn_du_doan(X_train, y_train, ma_train, X_moi, k, sigma=None):
    du_doan, ti_le, lang_gieng = [], [], []
    for q in np.atleast_2d(X_moi):
        # TODO 1: tính d2 tới mọi điểm train (chia hiệu cho sigma nếu sigma khác None)
        hieu = X_train - q
        if sigma is not None:
           hieu = hieu / sigma
        # Binh phuong khoang cach Euclid
        d2 = np.sum(hieu ** 2, axis=1)

        # TODO 2: top = k chỉ số đầu sau khi xếp theo (d2, ma_train)
        thu_tu = np.lexsort((ma_train, d2))
        top = thu_tu[:k]

        # TODO 3: đếm phiếu spam; thêm dự đoán, tỷ lệ phiếu và ma_train[top] vào ba danh sách
        phieu_spam = np.sum(y_train[top])
        du_doan.append(int(phieu_spam > k/2))
        ti_le.append(phieu_spam/k)
        lang_gieng.append(ma_train[top])

    return np.array(du_doan), np.array(ti_le), np.array(lang_gieng)

In [11]:
# TỰ KIỂM · bước 3: ví dụ năm điểm của slide
P = np.array([[0, 0], [2, 0], [0, 3], [4, 4], [1, 4]], float); nhan_P = np.array([0, 1, 1, 0, 0]); ma_P = np.arange(1, 6)
p1, _, lg1 = knn_du_doan(P, nhan_P, ma_P, [[1, 1]], 1)
p3, t3, lg3 = knn_du_doan(P, nhan_P, ma_P, [[1, 1]], 3)
p5, _, _ = knn_du_doan(P, nhan_P, ma_P, [[1, 1]], 5)
kiem('knn_du_doan trả về đủ ba mảng, mỗi tin mới một hàng', len(lg3) == 1 and len(lg3[0]) == 3, 'Mỗi lần lặp thêm một phần tử vào cả ba danh sách.')
kiem('X = (1; 1), k = 3: láng giềng P1, P2, P3 → spam (2/3 phiếu)', list(lg3[0]) == [1, 2, 3] and p3[0] == 1 and np.isclose(t3[0], 2 / 3),
     'd² = 2; 2; 5; 18; 9. Mỗi láng giềng một phiếu.')
kiem('X = (1; 1), k = 1: P1 trước P2 (cùng d² = 2) → ham', list(lg1[0]) == [1] and p1[0] == 0,
     'Đồng d² thì mã nhỏ hơn đứng trước: np.lexsort((ma_train, d2)), không dùng np.argsort mặc định.')
kiem('X = (1; 1), k = 5: ham (lớp đông thắng)', p5[0] == 0, '3 ham : 2 spam.')
p12, _, lg12 = knn_du_doan(P, nhan_P, ma_P, [[1, 2]], 3)
kiem('X = (1; 2), k = 3: P3, P5, P1 → ham (đồng d² ở ranh giới)', list(lg12[0]) == [3, 5, 1] and p12[0] == 0,
     'd² = 5; 5; 2; 13; 4: P1 và P2 cùng 5 tranh suất thứ ba.')

✅ knn_du_doan trả về đủ ba mảng, mỗi tin mới một hàng
✅ X = (1; 1), k = 3: láng giềng P1, P2, P3 → spam (2/3 phiếu)
✅ X = (1; 1), k = 1: P1 trước P2 (cùng d² = 2) → ham
✅ X = (1; 1), k = 5: ham (lớp đông thắng)
✅ X = (1; 2), k = 3: P3, P5, P1 → ham (đồng d² ở ranh giới)


## Bước 4 · Chuẩn hoá theo train (Chặng 3)

Tính μ và σ của **từng cột trên train** (σ chia cho n, như `StandardScaler`, tức `np.std(..., axis=0)` mặc định). Dùng `sigma` này cho mọi tập về sau.

Sau đó theo dõi tin 4500 (thuộc validation): ba láng giềng với Euclid thô và sau chuẩn hoá.

In [12]:
mu = X_tr.mean(axis=0)      # TODO: trung bình từng cột, chỉ trên X_tr
sigma = X_tr.std(axis=0, ddof=0)   # TODO: độ lệch chuẩn từng cột (chia n), chỉ trên X_tr

q4500 = d.loc[d.source_record == 4500, FEATURES].to_numpy()
p_tho, _, lg_tho = knn_du_doan(X_tr, y_tr, ma_tr, q4500, 3)
p_cn, _, lg_cn = knn_du_doan(X_tr, y_tr, ma_tr, q4500, 3, sigma=sigma)
print('μ =', mu.round(4), '· σ =', sigma.round(4))
print('Tin 4500 · thô:', lg_tho[0], '→', ['ham', 'spam'][p_tho[0]], '· chuẩn hoá:', lg_cn[0], '→', ['ham', 'spam'][p_cn[0]])

μ = [7.91869e+01 1.79000e-02] · σ = [5.89883e+01 4.59000e-02]
Tin 4500 · thô: [2780  626  646] → ham · chuẩn hoá: [4798 5042 4904] → spam


In [13]:
# TỰ KIỂM · bước 4 (mốc M2)
kiem('μ và σ của train', np.allclose(mu, [79.3102, 0.018052], atol=1e-4) and np.allclose(sigma, [59.1040, 0.047065], atol=1e-4),
     'Chỉ dùng X_tr; σ chia cho n (ddof = 0).')
kiem('Tin 4500 · thô: 2780, 626, 646 → ham', list(lg_tho[0]) == [2780, 626, 646] and p_tho[0] == 0, 'Không truyền sigma cho Euclid thô.')
kiem('Tin 4500 · chuẩn hoá: 4798, 3969, 2353 → spam', list(lg_cn[0]) == [4798, 3969, 2353] and p_cn[0] == 1, 'Truyền sigma của train.')

AssertionError: ❌ μ và σ của train. Gợi ý: Chỉ dùng X_tr; σ chia cho n (ddof = 0).

Ba láng giềng thô của tin 4500 đều dài 123 ký tự vì khi chưa chuẩn hóa, đặc trưng n_chars có thang đo lớn và chi phối phần lớn khoảng cách Euclid . Không được tính μ, σ trên toàn bộ 5159 tin vì sẽ sử dụng thông tin từ Validation và Test trong quá trình chuẩn hóa, gây rò rỉ dữ liệu và làm kết quả đánh giá thiếu khách quan . Vì vậy, μ và σ chỉ được tính trên tập Train.

**Câu hỏi M2:** vì sao ba láng giềng thô của tin 4500 đều dài đúng 123 ký tự? Vì sao không được tính μ, σ trên toàn bộ 5.159 tin?

## Bước 5 · Chọn cấu hình trên validation (Chặng 4)

Ứng viên **khai báo trước**: hai cách đo (thô, chuẩn hoá) × k ∈ {3; 5; 11}. Tiêu chí: **ít tin sai nhất trên validation**; hoà thì chọn k nhỏ hơn. Mức cơ bản đặt `LAM_BUOC_5 = False`: dùng cấu hình chuẩn hoá, k = 3.

In [14]:
LAM_BUOC_5 = True   # Mức cơ bản: đặt False; notebook vẫn chạy được đến cuối
UNG_VIEN = [(chuan_hoa, k) for chuan_hoa in (False, True) for k in (3, 5, 11)]
if LAM_BUOC_5:
    # TODO: với mỗi (chuan_hoa, k) trong UNG_VIEN, dự đoán X_va và đếm số tin sai; lập bảng `bang`
    # với các cột 'chuẩn hoá', 'k', 'tin sai'; rồi CHON = (chuan_hoa, k) có ít tin sai nhất (hoà → k nhỏ hơn)
    ket_qua=[]

    for chuan_hoa, k in UNG_VIEN:
      if chuan_hoa:
        X_train = (X_tr - mu) / sigma
        X_val = (X_va - mu) / sigma
      else:
        X_train = X_tr
        X_val = X_va

      du_doan,_, _ = knn_du_doan(
          X_train,y_tr,ma_tr,X_val,k
      )

      tin_sai = np.sum(du_doan != y_va)

      ket_qua.append({
          'chuẩn hóa' : chuan_hoa,
          'k' : k,
          'tin sai': tin_sai
      })

    bang = pd.DataFrame(ket_qua)

    bang = bang.sort_values(
        by=['tin sai' , 'k'],
        ascending=[True,True]
    ).reset_index(drop=True)

    hang_tot_nhat = bang.iloc[0]
    CHON = (bool(hang_tot_nhat['chuẩn hóa']), int(hang_tot_nhat['k']))

    print(bang.to_string(index=False))
else:
    CHON = (True, 3)
print('Cấu hình khoá:', 'chuẩn hoá' if CHON[0] else 'thô', '· k =', CHON[1])

 chuẩn hóa  k  tin sai
      True 11       31
      True  3       36
      True  5       38
     False  3       53
     False  5       62
     False 11       76
Cấu hình khoá: chuẩn hoá · k = 11


In [15]:
# TỰ KIỂM · bước 5
if LAM_BUOC_5:
    kiem('Bảng số tin sai trên validation: thô 52/54/80, chuẩn hoá 29/29/25', list(bang['tin sai']) == [52, 54, 80, 29, 29, 25],
         'Dùng đúng quy ước đồng hạng; chuẩn hoá bằng sigma của TRAIN; chấm trên validation.')
    kiem('Chọn chuẩn hoá, k = 11', CHON == (True, 11), 'Ít tin sai nhất; hoà thì k nhỏ hơn.')
else:
    print('⏭ Bỏ qua bước 5 (mức cơ bản): dùng chuẩn hoá, k = 3')

AssertionError: ❌ Bảng số tin sai trên validation: thô 52/54/80, chuẩn hoá 29/29/25. Gợi ý: Dùng đúng quy ước đồng hạng; chuẩn hoá bằng sigma của TRAIN; chấm trên validation.

## Bước 6 · Khoá, chấm test một lần, so mốc, đọc tin sai (Chặng 4)

`cham_test` chỉ cho chấm **một lần**. Mốc "luôn đoán ham" chỉ để đối chiếu, không dùng để chọn.

In [17]:
_SO_LAN_CHAM = globals().get('_SO_LAN_CHAM', 0)
def cham_test(du_doan):
    global _SO_LAN_CHAM
    if _SO_LAN_CHAM >= 1:
        raise RuntimeError('Test đã được chấm. Muốn đổi cấu hình: quay lại validation và ghi rõ trong báo cáo.')
    _SO_LAN_CHAM += 1
    tn = int(((du_doan == 0) & (y_te == 0)).sum()); fp = int(((du_doan == 1) & (y_te == 0)).sum())
    fn = int(((du_doan == 0) & (y_te == 1)).sum()); tp = int(((du_doan == 1) & (y_te == 1)).sum())
    return tn, fp, fn, tp

p_te, tl_te, lg_te = knn_du_doan(X_tr, y_tr, ma_tr, X_te, CHON[1], sigma=sigma if CHON[0] else None)
tn, fp, fn, tp = cham_test(p_te)
print(f'Test · {"chuẩn hoá" if CHON[0] else "thô"}, k = {CHON[1]}:  TN {tn} · FP {fp} (ham bị chặn nhầm) · FN {fn} (spam bị bỏ lọt) · TP {tp}')
print(f'Tổng {fp + fn} tin sai / {len(y_te)} · mốc "luôn ham": {int(y_te.sum())} tin sai (đúng {100 * (1 - y_te.mean()):.1f}%)'.replace('.', ','))

Test · chuẩn hoá, k = 11:  TN 895 · FP 9 (ham bị chặn nhầm) · FN 13 (spam bị bỏ lọt) · TP 115
Tổng 22 tin sai / 1032 · mốc "luôn ham": 128 tin sai (đúng 87,6%)


In [18]:
# TỰ KIỂM · quy trình (mốc M3)
kiem('cham_test được gọi đúng một lần', _SO_LAN_CHAM == 1, 'Chạy lại toàn bộ notebook (Restart & Run All) nếu đã gọi nhiều lần.')
kiem('Tốt hơn mốc "luôn ham"', fp + fn < int(y_te.sum()), 'Nếu không tốt hơn mốc, kiểm lại chuẩn hoá và quy ước đồng hạng.')

✅ cham_test được gọi đúng một lần
✅ Tốt hơn mốc "luôn ham"


In [20]:
# Đọc ba tin sai: chọn ít nhất một tin bị chặn nhầm và một tin bị bỏ lọt; ghi nhận xét trong báo cáo
bang_te = d.loc[te, ['source_record', 'label', 'n_chars', 'digit_fraction', 'text']].assign(du_doan=np.where(p_te == 1, 'spam', 'ham'), ti_le_phieu=tl_te)
sai = bang_te[bang_te.label != bang_te.du_doan]
# Chon 1 tin ham bi du doan nham thanh spam
fp_mau=sai[
    (sai['label'] == 'ham') & (sai['du_doan'] == 'spam')
].head(1)

# Chon 1 tin spam bi du doan nham thanh ham
fn_mau = sai[
    (sai['label'] == 'spam') & (sai['du_doan'] == 'ham')
].head(1)

# Chon them 1 tin sai bat ky
con_lai = sai.drop(
    fp_mau.index.union(fn_mau.index)
).head(1)

ba_tin = pd.concat([fp_mau, fn_mau, con_lai])
pd.set_option('display.max_colwidth', 90)
ba_tin

,source_record,label,n_chars,digit_fraction,text,du_doan,ti_le_phieu
510,511,ham,136,0.066176,Your opinion about me? 1. Over 2. Jada 3. Kusruthi 4. Lovable 5. Silent 6. Spl charact...,spam,1.000000
4249,4250,spam,74,0.000000,"accordingly. I repeat, just text the word ok on your mobile phone and send",ham,0.000000
4728,4729,spam,143,0.013986,Had your mobile 10 mths? Update to the latest Camera/Video phones for FREE. KEEP UR SA...,ham,0.363636


### Đối chiếu với scikit-learn (mức khá)

Dùng `Pipeline(StandardScaler, KNeighborsClassifier)` với đúng cấu hình đã khoá **trên test đã chấm** (chỉ để đối chiếu cài đặt, không để chọn lại). Nếu lệch vài tin, hãy kiểm các tin đó có **đồng d² ở ranh giới thứ k** không: scikit-learn không xếp đồng hạng theo mã bản ghi, và kết quả của nó phụ thuộc thứ tự dòng của train (slide 49). Giữ nguyên thứ tự dòng do `train_test_split` trả về.

In [21]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
import sklearn
if CHON[0]:
    mo_hinh = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=CHON[1], algorithm='brute'))
else:
    mo_hinh = KNeighborsClassifier(n_neighbors=CHON[1], algorithm='brute')
mo_hinh.fit(X_tr, y_tr)
p_sk = mo_hinh.predict(X_te)
khop = int((p_sk == p_te).sum())
print(f'scikit-learn {sklearn.__version__}: khớp {khop}/{len(p_te)} dự đoán test')
kiem('Cài đặt tay khớp thư viện, sai khác tối đa 5 tin', khop >= len(p_te) - 5,
     'Nếu lệch nhiều: kiểm công thức d², sigma của train và cách đếm phiếu. Lệch vài tin thường do đồng d² ở ranh giới.')

scikit-learn 1.6.1: khớp 1032/1032 dự đoán test
✅ Cài đặt tay khớp thư viện, sai khác tối đa 5 tin


## Báo cáo một trang (nộp cùng notebook)

Mỗi mục 2–4 câu.

1. **Bài toán và dữ liệu.** Dự đoán gì, dữ liệu nào, đã bỏ trùng thế nào; số tin và số spam mỗi tập.
2. **Phương pháp.** Hai đặc trưng; quy ước kNN và đồng hạng; chuẩn hoá theo train.
3. **Kết quả.** Bảng validation; cấu hình đã khoá; bảng test (TN, FP, FN, TP) so với mốc "luôn ham".
4. **Ba tin sai.** Vì sao hai đặc trưng dẫn kNN tới kết quả sai ở từng tin (mô tả, không suy diễn nhân quả).
5. **Giới hạn.** Ít nhất ba: đặc trưng chỉ đo hình thức; corpus tiếng Anh cũ, chia ngẫu nhiên; tỷ lệ phiếu không phải xác suất; một lần chia.
6. **Việc làm tiếp.** Một thay đổi bạn sẽ thử và cách kiểm nó **không** dùng test.

## Nâng cao (tuỳ chọn, mọi lựa chọn chỉ trên validation)

- **Đặc trưng thứ ba:** tỷ lệ chữ in hoa `sum(c.isupper()) / len(text)`. Mở rộng `knn_du_doan` cho 3 cột; so số tin sai trên validation.
- **MinMax thay cho z-score:** min, max lấy từ train; tin mới có thể vượt khỏi [0; 1].
- **Bỏ phiếu theo khoảng cách:** trọng số 1/d, có quy ước khi d = 0.
- **Gợi ý xa hơn:** TF-IDF trên văn bản (Bài 2 chưa dạy); chỉ so trên validation, giữ test cho phép chấm cuối.